# In-silico endpoint mapping / computational–clinical translation layer

**This notebook is not a clinical trial, not FDA/EMA readiness, and not a Phase II readout.**

It maps trajectories from the **real** Confluence closed loop
(`CancerODE` / observation layer / PK/PD / controllers A, B, E, F)
onto RECIST 1.1-*like* response, CTCAE v5.0-*like* grades, Kaplan–Meier OS/PFS,
and computed log-rank / Cox HR + 95% CI.

p-values and hazard ratios are **never hardcoded**. If the virtual cohort is
underpowered, the suite reports the confidence interval and marks the comparison
**inconclusive**.

See `DISCLAIMER.md` and `confluence/benchmarks/clinical_endpoint_mapper.py`.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

from confluence.benchmarks.clinical_endpoint_mapper import ENDPOINT_NON_CLAIM
from confluence.benchmarks.closed_loop_translation import (
    four_panel_figure,
    run_translation,
)

print(ENDPOINT_NON_CLAIM)
assert "not a clinical trial" in ENDPOINT_NON_CLAIM.lower()

## Run the translation layer on real A/B/E/F arms

The committed figure in `results/validation_translation/` is produced with
`N=100` Latin-hypercube virtual patients (each patient × four controllers).
This cell uses a smaller `N` so the notebook stays interactive; set `N=100`
to reproduce the robustness cohort.

CLI equivalent:

```bash
python3 -m confluence.benchmarks.closed_loop_translation --n 100 --out results/validation_translation
python3 -m confluence.benchmarks.validation_suite --clinical --clinical-n 100
```

In [ ]:
OUT = Path("results/validation_translation")
committed = OUT / "four_panel_endpoints.png"
if committed.exists():
    report = None
    print("Using committed N≥100 four-panel figure at", committed)
else:
    report = run_translation(n=8, out_dir=OUT, days=16.0, seed=17)
    print("stiff agreed", report["part1_stiff_solver"]["agreed"])
    print("conservation", report["part1_conservation"]["ok"])
    print("weights bounded", report["part1_weight_convergence"]["bounded"])
    print("pairwise", report["part2_cohort"]["pairwise"])

## Four-panel figure (KM, RECIST, CTCAE, example trajectory)

Generated from real closed-loop arms, not a standalone Euler script.

In [ ]:
fig_path = Path("results/validation_translation/four_panel_endpoints.png")
if fig_path.exists():
    display(Image(filename=str(fig_path)))
elif report is not None:
    four_panel_figure(report["part2_cohort"], fig_path)
    display(Image(filename=str(fig_path)))
else:
    raise FileNotFoundError("Run run_translation(...) first")

## Honesty recap

- Simulated endpoints for scientific scrutiny of the ODE + controller.
- Q3W anti-PD-1 pulses and 5/2 TKI/HDAC holidays are **simulated regimens**.
- Do not read CR/PR or a log-rank p-value as a clinical claim.